# Customer Churn Prediction & LTV Engine
## Notebook 04: SQL & Feature Engineering Pipeline

**Owner**: Abhishek (SQL & Feature Engineering)  
**Project**: Customer Churn Prediction & LTV Engine  
**Purpose**: Clean raw telecommunication data, construct domain-driven engineered features (tenure, billing, bundling, friction, and interactions), implement behavioral customer segmentation, and prepare leakage-free model-ready datasets for Churn (classification) and LTV (regression) models.

### 1. Environment Setup and Library Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path to import src
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.feature_engineering import (
    clean_raw_data,
    engineer_features,
    CustomerFeatureEngineer,
    prepare_model_dataset
)

# Plotting style configurations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 10

print(f"Project root resolved: {project_root}")

### 2. Ingesting Raw Data and Inspecting Hygiene Issues

In [ ]:
raw_data_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Raw Dataset Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
# Inspect data types and detect whitespace-nulls in TotalCharges
print("Total null entries detected by pandas:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

empty_total_charges = df_raw[df_raw["TotalCharges"].astype(str).str.strip() == ""]
print(f"Blank/whitespace TotalCharges rows: {len(empty_total_charges)}")
print("Tenure of blank TotalCharges rows:", empty_total_charges["tenure"].unique())

### 3. Data Cleaning & Sanitization

In [ ]:
df_cleaned = clean_raw_data(df_raw)
print(f"Cleaned dataset shape: {df_cleaned.shape}")
print(f"TotalCharges dtype: {df_cleaned['TotalCharges'].dtype}")
print(f"Overall Churn Rate: {df_cleaned['is_churned'].mean():.2%}")

### 4. Domain Feature Engineering
We extract rich behavioral, financial, and engagement signals across:
1. **Tenure Features**: Log tenure, tenure cohorts (`0-6m`, `7-12m`, etc.), lifecycle flags.
2. **Charges & Monetary Signals**: Charges ratio (actual vs expected tenure charges), historical avg monthly charge, charge velocity.
3. **Contract & Friction**: Month-to-month flag, auto-pay protection, paperless manual payment risk.
4. **Service Bundling & Cross-sell**: Active service count, streaming bundle, security bundle, fiber optic without tech support risk.
5. **Customer Segmentation & LTV**: Value tiers, behavioral risk categories, strategic action matrix.

In [ ]:
df_engineered = engineer_features(df_cleaned)
print(f"Engineered Dataset Shape: {df_engineered.shape}")
print("Sample Engineered Features:")
cols_preview = [
    "customerID", "tenure_cohort", "charges_ratio", "charge_velocity",
    "total_services_count", "has_fiber_no_techsupport_risk",
    "value_tier", "risk_profile", "customer_strategic_segment", "estimated_total_ltv"
]
df_engineered[cols_preview].head(10)

### 5. Exploratory Visualizations & Feature Impact Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# A. Churn Rate by Tenure Cohort
cohort_churn = df_engineered.groupby("tenure_cohort", observed=False)["is_churned"].mean().reset_index()
sns.barplot(data=cohort_churn, x="tenure_cohort", y="is_churned", ax=axes[0], palette="Blues_r")
axes[0].set_title("Churn Rate by Tenure Cohort", fontsize=13, weight="bold")
axes[0].set_ylabel("Churn Rate")
axes[0].set_xlabel("Tenure Cohort")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# B. Churn Rate by Contract Type
contract_churn = df_engineered.groupby("Contract", observed=False)["is_churned"].mean().reset_index()
sns.barplot(data=contract_churn, x="Contract", y="is_churned", ax=axes[1], palette="Reds_r")
axes[1].set_title("Churn Rate by Contract Commitment", fontsize=13, weight="bold")
axes[1].set_ylabel("Churn Rate")
axes[1].set_xlabel("Contract Type")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# C. Impact of Fiber Optic without Tech Support Risk Flag
fiber_churn = df_engineered.groupby("has_fiber_no_techsupport_risk")["is_churned"].mean().reset_index()
fiber_churn["Label"] = fiber_churn["has_fiber_no_techsupport_risk"].map({0: "Normal / Supported", 1: "Fiber w/o TechSupport"})
sns.barplot(data=fiber_churn, x="Label", y="is_churned", ax=axes[0], palette="Set2")
axes[0].set_title("Key Risk Interaction: Fiber w/o Tech Support", fontsize=13, weight="bold")
axes[0].set_ylabel("Churn Rate")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

# D. Churn Rate by Active Service Breadth Count
services_churn = df_engineered.groupby("total_services_count")["is_churned"].mean().reset_index()
sns.barplot(data=services_churn, x="total_services_count", y="is_churned", ax=axes[1], palette="viridis")
axes[1].set_title("Service Breadth (Total Services) vs Churn Rate", fontsize=13, weight="bold")
axes[1].set_ylabel("Churn Rate")
axes[1].set_xlabel("Total Active Services")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

plt.tight_layout()
plt.show()

### 6. Customer Strategic Segmentation Analysis

In [ ]:
seg_summary = df_engineered.groupby("customer_strategic_segment").agg(
    customer_count=("customerID", "count"),
    churn_rate=("is_churned", "mean"),
    arpu=("MonthlyCharges", "mean"),
    avg_historical_ltv=("TotalCharges", "mean"),
    avg_projected_ltv=("estimated_total_ltv", "mean")
).reset_index()

seg_summary["churn_rate_pct"] = seg_summary["churn_rate"].map(lambda x: f"{x:.2%}")
seg_summary["arpu"] = seg_summary["arpu"].round(2)
seg_summary["avg_historical_ltv"] = seg_summary["avg_historical_ltv"].round(2)
seg_summary["avg_projected_ltv"] = seg_summary["avg_projected_ltv"].round(2)

print("Strategic Customer Segments Performance Table:")
seg_summary[["customer_strategic_segment", "customer_count", "churn_rate_pct", "arpu", "avg_historical_ltv", "avg_projected_ltv"]]

### 7. Feature Correlation Heatmap with Churn Target

In [ ]:
numeric_features = [
    "tenure_months", "log_tenure", "MonthlyCharges", "TotalCharges",
    "charges_ratio", "avg_historical_monthly_charge", "charge_velocity",
    "contract_term_months", "is_month_to_month", "is_autopay_enabled",
    "total_services_count", "has_fiber_no_techsupport_risk", "is_churned"
]

corr_matrix = df_engineered[numeric_features].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Correlation Matrix: Engineered Features vs Churn", fontsize=14, weight="bold")
plt.tight_layout()
plt.show()

### 8. Leakage-Free Model Dataset Preparation & Export
We use `prepare_model_dataset` to:
1. Perform a stratified train/test split on `is_churned`.
2. Fit scalers and one-hot encoders strictly on the training partition.
3. Transform the test partition without data leakage.
4. Export `X_train.csv`, `X_test.csv`, `y_churn_train.csv`, `y_churn_test.csv`, `y_ltv_train.csv`, `y_ltv_test.csv`, and `feature_metadata.json` for Varsha's modeling notebooks.

In [ ]:
output_dir = project_root / "data" / "processed"
dataset_dict = prepare_model_dataset(
    raw_csv_path=raw_data_path,
    test_size=0.2,
    random_state=42,
    output_dir=output_dir
)

X_train = dataset_dict["X_train"]
X_test = dataset_dict["X_test"]
y_churn_train = dataset_dict["y_churn_train"]
y_churn_test = dataset_dict["y_churn_test"]
y_ltv_train = dataset_dict["y_ltv_train"]
y_ltv_test = dataset_dict["y_ltv_test"]

print("=" * 60)
print("MODEL-READY DATASET EXPORT SUMMARY:")
print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")
print(f"y_churn_train: {y_churn_train.shape} (Mean Churn: {y_churn_train.mean():.2%})")
print(f"y_churn_test: {y_churn_test.shape} (Mean Churn: {y_churn_test.mean():.2%})")
print(f"y_ltv_train: {y_ltv_train.shape} (Mean LTV: ${y_ltv_train.mean():.2f})")
print(f"y_ltv_test: {y_ltv_test.shape} (Mean LTV: ${y_ltv_test.mean():.2f})")
print(f"Files saved to: {output_dir}")
print("=" * 60)

### 9. Verification for Downstream Modeling Notebooks
Verify that the model-ready dataset can be consumed directly by `notebooks/05_churn_modeling.ipynb` and `notebooks/06_ltv_modeling.ipynb`.

In [ ]:
# Verify no missing values in processed features
assert X_train.isnull().sum().sum() == 0, "NaNs found in X_train!"
assert X_test.isnull().sum().sum() == 0, "NaNs found in X_test!"
print("[PASSED] Zero missing values in train and test feature sets.")

# Quick sanity baseline classification fit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

baseline_clf = LogisticRegression(max_iter=500)
baseline_clf.fit(X_train, y_churn_train)
test_preds = baseline_clf.predict(X_test)
test_probs = baseline_clf.predict_proba(X_test)[:, 1]

print(f"Baseline Logistic Regression Accuracy: {accuracy_score(y_churn_test, test_preds):.4f}")
print(f"Baseline ROC-AUC Score: {roc_auc_score(y_churn_test, test_probs):.4f}")
print("Feature engineering pipeline is fully ready for churn and LTV modeling!")